# Step 01 — Chapter 2: The Data Pipeline & Tokenization

Packages for this chapter.

In [40]:
!pip install numpy requests torch tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.2/15.2 MB 65.5 MB/s eta 0:00:0000:0100:01


In [2]:
from importlib.metadata import version

print("requests version:", version("requests"))
print("tiktoken version:", version("tiktoken"))
print("torch version:", version("torch"))


requests version: 2.34.2
tiktoken version: 0.13.0
torch version: 2.13.0


### 01.1. Download text


[The Verdict by Edith Wharton](https://en.wikisource.org/wiki/The_Verdict) is a public domain short story we will use as the raw text to work with.

In [3]:
import os
import requests

# download the raw text in case we do not have it locally yet
if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)

with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


### 01.2. Tokenize text

In [4]:
import re

# naively split the text by whitespace
print("Simple split")
result = re.split(r'(\s)', raw_text)
result = [item.strip() for item in result if item.strip()]
print(result[:15])
print(len(result))

# tokenize the raw text
print("\nTokenized")
tokenized = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
tokenized = [item.strip() for item in tokenized if item.strip()]
print(tokenized[:15])
print(len(tokenized))

Simple split
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius--though', 'a', 'good', 'fellow', 'enough--so', 'it']
3634

Tokenized
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow']
4690


### 01.3. Convert to token IDs

In [5]:
# find all unique tokens
tokens = sorted(set(tokenized))
tokens_size = len(tokens)
print(tokens_size)

# build the vocabulary
print("\nVocabulary")
vocab = {token:id for id,token in enumerate(tokens)}
for id, item in enumerate(vocab.items()):
    print(item)
    if id >= 15:
        break

1130

Vocabulary
('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)


In [6]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {id:token for token,id in vocab.items()}
    
    def encode(self, text):
        tokenized = re.split(r'([,.:;?_!"()\']|--|\s)', text)
                                
        tokenized = [item.strip() for item in tokenized if item.strip()]
        ids = [self.str_to_int[id] for id in tokenized]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[id] for id in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [7]:
tokenizer = SimpleTokenizerV1(vocab)

print("Tokenized")
print(tokenized[:10])

print("\nEncoded token IDs")
ids = tokenizer.encode(raw_text)
print(ids[:10])

print("\nDecoded token IDs")
tokens = tokenizer.decode(ids[:10])
print(tokens)

Tokenized
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius']

Encoded token IDs
[53, 44, 149, 1003, 57, 38, 818, 115, 256, 486]

Decoded token IDs
I HAD always thought Jack Gisburn rather a cheap genius


### 01.4. Add special context tokens

In [8]:
tokenizer = SimpleTokenizerV1(vocab)

text = "Hello, do you like tea. Is this-- a test?"

# NOTE: THIS WILL FAIL as we do not have a token for "Hello"
tokenizer.encode(text)

KeyError: 'Hello'

In [12]:
extended_tokens = sorted(list(set(tokenized)))
# add an extra <|unk|> token for words we do not have in the vocabulary
extended_tokens.extend(["<|unk|>"])

extended_vocab = {token:id for id,token in enumerate(extended_tokens)}

# check the new length of of our vocabulary 1130 + 1 = 1131
print(len(extended_vocab))

1131


In [14]:
# An updated Tokenizer that encodes for unknown tokens
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { id:token for token,id in vocab.items()}
    
    def encode(self, text):
        tokenized = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        tokenized = [item.strip() for item in tokenized if item.strip()]
        tokenized = [
            item if item in self.str_to_int 
            else "<|unk|>" for item in tokenized
        ]

        ids = [self.str_to_int[token] for token in tokenized]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [16]:
tokenizer = SimpleTokenizerV2(extended_vocab)

print("\nEncoded token IDs")
ids = tokenizer.encode(text)
print(ids)

print("\nDecoded token")
tokens = tokenizer.decode(ids)
print(tokens)


Encoded token IDs
[1130, 5, 355, 1126, 628, 975, 7, 1130, 999, 6, 115, 1130, 10]

Decoded token
<|unk|>, do you like tea. <|unk|> this -- a <|unk|>?


### 01.5. BytePair encoding

- GPT-2 used BytePair encoding (BPE) as its tokenizer to encode words that aren't part of its predefined vocabulary.
- It breaks down words into smaller subword units or individual characters so it can process words not contained in its vocabulary.
- GPT-2's original BPE tokenizer is avaialble at https://github.com/openai/gpt-2/blob/master/src/encoder.py
- However, the more performant choice is to use OpenAI's BPE [tiktoken](https://github.com/openai/tiktoken) tokenizer as it implements the core algorithm in Rust.

In [18]:
import tiktoken
print("tiktoken version:", version("tiktoken"))

tiktoken version: 0.13.0


In [21]:
tokenizer = tiktoken.get_encoding("gpt2")

text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

# Use the tiktoken GPT-2 tokenizer to encode the text
ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(ids)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [23]:
# Reverse the process to validate it works as expected
words = tokenizer.decode(ids)

print(words)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


### 01.6. Sliding window data sampling

In [27]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print("Encoded text:")
print(enc_text[:15])
print("\nEncoded text length:")
print(len(enc_text))

Encoded text:
[40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138, 257, 7026, 15632, 438, 2016]

Encoded text length:
5145


In [36]:
enc_example = enc_text[50:]
context_size = 4

# Predicting the next word can be trained by shifting
# the context window inputs by one position to the right.
x = enc_example[:context_size]
y = enc_example[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


In [37]:
# Applying this approach we can produce the desired value
for i in range(1, context_size+1):
    context = enc_example[:i]
    desired = enc_example[i]

    print(context, " ---> ", desired)

[290]  --->  4920
[290, 4920]  --->  2241
[290, 4920, 2241]  --->  287
[290, 4920, 2241, 287]  --->  257


In [38]:
# Or in the decoded version
for i in range(1, context_size+1):
    context = enc_example[:i]
    desired = enc_example[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


In [43]:
import torch
from torch.utils.data import Dataset, DataLoader

print("PyTorch version:", torch.__version__)

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

PyTorch version: 2.13.0+cu130


In [44]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [ ]:
# Test the dataloader to get the intuition
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print("First batch:")
print(first_batch)

second_batch = next(data_iter)
print("\nSecond batch:")
print(second_batch)

First batch:
[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]

Second batch:
[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [49]:
# Two changes to make this more realistic 
# 1) Increase the batch size to 8
# 2) Match the stride to max_length to avoid overlap
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


### 01.7. Create Token Embeddings

In [ ]:
# Simple input with 4 Token IDs
input_ids = torch.tensor([2, 3, 5, 1])

# Params for our Embedding layer (6x3 weight matrix)
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [ ]:
# Embed Token ID 3 into the 3-dimensional vector
# Should match the 4th row in the matrix above

print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


In [59]:
# Embed the input_ids Token IDs from above
print("Embed the input_ids Token IDs")
print(input_ids)

print("\nOne by one:")
print(embedding_layer(input_ids[0]))
print(embedding_layer(input_ids[1]))
print(embedding_layer(input_ids[2]))
print(embedding_layer(input_ids[3]))

print("\nIn one go:")
print(embedding_layer(input_ids))

Embed the input_ids Token IDs
tensor([2, 3, 5, 1])

One by one:
tensor([ 1.2753, -0.2010, -0.1606], grad_fn=<EmbeddingBackward0>)
tensor([-0.4015,  0.9666, -1.1481], grad_fn=<EmbeddingBackward0>)
tensor([-2.8400, -0.7849, -1.4096], grad_fn=<EmbeddingBackward0>)
tensor([0.9178, 1.5810, 1.3010], grad_fn=<EmbeddingBackward0>)

In one go:
tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


### 01.8. Encode word positions

In [64]:
# Retrieve the Byte Pair Encoder's vocabulary size (50,257)
tokenizer = tiktoken.get_encoding("gpt2")
vocab_size = tokenizer.n_vocab 
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [65]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

print("Token IDs:\n", inputs)
print("\ninputs.shape:\n", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

inputs.shape:
 torch.Size([8, 4])


In [68]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
#print(token_embeddings)

torch.Size([8, 4, 256])


In [74]:
# Create other embedding layer for GPT-2 absolute position embeddings:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

# uncomment & execute the following line to see how the embedding layer weights look like
#print(pos_embedding_layer.weight)

pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
print(pos_embeddings)

torch.Size([4, 256])
tensor([[ 1.3680,  0.2229,  0.1775,  ...,  1.4890, -0.1371, -1.0218],
        [-0.0769, -1.2147,  0.2052,  ..., -0.0639, -0.5527, -0.8053],
        [ 0.0127,  0.2883, -0.2844,  ...,  0.1624,  0.0567,  1.1401],
        [ 1.8016, -1.4982, -1.4812,  ...,  0.0432,  0.0044,  0.7157]],
       grad_fn=<EmbeddingBackward0>)


In [76]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
#print(input_embeddings)

torch.Size([8, 4, 256])
